# AlphaFold 1 & 2：蛋白质结构预测

这个 Notebook 深入解析 `AlphaFold 1（2018）` 和 `AlphaFold 2（2021）` 的架构演进，展示蛋白质结构预测的核心原理与实际推理流程。

内容包括：
- 蛋白质结构预测问题的定义（序列 → 三维结构）
- AlphaFold 1：进化信息 + CNN + 距离矩阵预测
- AlphaFold 2：Evoformer + Structure Module 架构革命
- MSA（多序列比对）的作用与构建
- 使用 ColabFold/ESMFold 做轻量推理演示
- 结构可视化（pLDDT 置信度着色）
- AlphaFold 对生物学的范式意义

## 1. 环境准备

```bash
# 轻量推理用 ESMFold（Meta AI，无需 MSA，本地可运行）
pip install torch transformers biopython matplotlib numpy requests

# 3D 结构可视化（可选，需要 py3Dmol）
pip install py3Dmol ipywidgets
```

> AlphaFold 2 完整推理需要 MSA 检索（数百 GB 数据库），本 Notebook 使用 ESMFold 做本地演示，结构精度与 AF2 接近但无需 MSA。

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch
from transformers import EsmForProteinFolding, AutoTokenizer

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # ESMFold：Meta 的单序列蛋白质折叠模型，无需 MSA
    model_name: str = 'facebook/esmfold_v1'
    # 氨基酸序列长度限制（避免显存不足）
    max_length: int = 400

cfg = Config()

# 演示序列：人类 Ubiquitin（76 氨基酸，经典蛋白质折叠基准）
UBIQUITIN = 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG'
# 演示序列：Trp-cage（20 氨基酸，最小可折叠蛋白）
TRP_CAGE = 'NLYIQWLKDGGPSSGRPPPS'

print(f'Ubiquitin 序列长度：{len(UBIQUITIN)} aa')
print(f'Trp-cage 序列长度 ：{len(TRP_CAGE)} aa')

## 2. 问题定义：序列 → 结构

蛋白质由氨基酸序列决定，氨基酸序列决定三维结构，三维结构决定功能。

### 2.1 蛋白质结构层次

| 层次 | 描述 |
|------|------|
| 一级结构 | 氨基酸序列（A, C, D, E, ... 共 20 种） |
| 二级结构 | α-螺旋、β-折叠等局部构型 |
| 三级结构 | 完整的三维空间坐标（每个原子的 x, y, z） |
| 四级结构 | 多条链组成的复合物 |

### 2.2 为什么难？

一个长度为 100 的蛋白质，仅主链有约 200 个可旋转键，理论构型数量达到 $10^{40}$ 量级。
传统实验（X 射线晶体学、冷冻电镜）解析一个蛋白质结构需要数年时间和数百万美元。

## 3. AlphaFold 1 架构解读（2018）

### 3.1 核心思路

AlphaFold 1 把结构预测转化为**距离矩阵预测**问题：

$$\text{序列} \xrightarrow{\text{MSA + 共进化分析}} \text{残基对距离分布} \xrightarrow{\text{势能最小化}} \text{三维结构}$$

### 3.2 关键模块

| 模块 | 方法 | 作用 |
|------|------|------|
| 特征提取 | MSA（多序列比对） + PSICOV 共进化矩阵 | 利用进化压力推断哪些残基在三维空间上接近 |
| 主干网络 | 深度残差 CNN（220层） | 从共进化矩阵预测残基对距离分布 |
| 结构重建 | 势能最小化（梯度下降） | 从距离约束重建三维坐标 |

### 3.3 局限性

- 距离矩阵预测 → 结构重建 是两阶段，误差累积
- CNN 感受野有限，全局折叠信息利用不足
- CASP13 竞赛中 GDT 分数约 60，仅领先第二名约 15%

In [ ]:
def plot_af1_pipeline():
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 2)

    steps = [
        (0.5, '氨基酸\n序列'),
        (2.5, 'MSA\n共进化矩阵'),
        (4.5, '深度残差 CNN\n(220 层)'),
        (6.5, '距离/角度\n分布预测'),
        (8.5, '势能优化\n→ 3D 坐标'),
    ]
    colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

    for (x, label), color in zip(steps, colors):
        ax.add_patch(mpatches.FancyBboxPatch((x - 0.8, 0.4), 1.6, 1.2,
                     boxstyle='round,pad=0.1', facecolor=color, alpha=0.8))
        ax.text(x, 1.0, label, ha='center', va='center', fontsize=9, color='white', fontweight='bold')

    for i in range(len(steps) - 1):
        x1 = steps[i][0] + 0.8
        x2 = steps[i+1][0] - 0.8
        ax.annotate('', xy=(x2, 1.0), xytext=(x1, 1.0),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=2))

    ax.set_title('AlphaFold 1 流程（2018）', fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


plot_af1_pipeline()

## 4. AlphaFold 2 架构解读（2021）

AlphaFold 2 在 CASP14 上以平均 GDT 92.4 的成绩震惊生物学界，被评为「50 年来最重要的科学突破之一」。

### 4.1 两大核心模块

#### Evoformer（进化 Transformer）

输入：
- **MSA 矩阵**（$N_{seq} \times L$ 对齐序列）
- **Pair 表示**（$L \times L$ 残基对特征）

关键操作：
- **行注意力**：同一列（同一位置）的不同序列之间交互，捕捉进化约束
- **列注意力**：同一序列中不同位置之间交互
- **Outer Product Mean**：将 MSA 信息注入 Pair 表示
- **Triangle Attention**：Pair 表示内的三角更新，强制几何一致性（若 i-j 近、j-k 近 → i-k 也近）

#### Structure Module（结构模块）

- 直接预测每个残基的**刚体变换**（旋转 + 平移），而非距离矩阵
- **IPA（Invariant Point Attention）**：在三维空间中做注意力，保持旋转/平移不变性
- **端到端可微分**：结构模块的输出梯度可以流回 Evoformer

### 4.2 AF1 → AF2 关键进步

| 维度 | AlphaFold 1 | AlphaFold 2 |
|------|-------------|-------------|
| 主干 | CNN | Transformer（Evoformer） |
| 输出 | 距离分布（间接） | 原子坐标（直接） |
| 结构重建 | 后处理势能优化 | 端到端可微 Structure Module |
| 模板利用 | 有限 | 显式模板注意力机制 |
| CASP 成绩 | ~60 GDT | ~92 GDT（接近实验精度） |

In [ ]:
def plot_af2_pipeline():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 左：Evoformer 示意
    ax = axes[0]
    ax.set_xlim(0, 6)
    ax.set_ylim(0, 8)

    blocks = [
        (3, 7.0, 'MSA 行列注意力', '#4C72B0'),
        (3, 5.5, 'Outer Product Mean', '#DD8452'),
        (3, 4.0, 'Triangle Attention\n(Pair 表示)', '#55A868'),
        (3, 2.5, 'Triangle MLP Update', '#C44E52'),
        (3, 1.0, '× 48 层', '#8172B2'),
    ]
    for x, y, label, color in blocks:
        ax.add_patch(mpatches.FancyBboxPatch((x-2.2, y-0.5), 4.4, 0.9,
                     boxstyle='round,pad=0.05', facecolor=color, alpha=0.8))
        ax.text(x, y, label, ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    ax.set_title('Evoformer（进化 Transformer）', fontsize=11)
    ax.axis('off')

    # 右：Structure Module 示意
    ax = axes[1]
    ax.set_xlim(0, 6)
    ax.set_ylim(0, 8)

    blocks2 = [
        (3, 7.0, 'IPA（不变点注意力）', '#4C72B0'),
        (3, 5.5, '刚体变换更新\n（旋转 + 平移）', '#DD8452'),
        (3, 4.0, '骨架原子坐标', '#55A868'),
        (3, 2.5, '侧链扭转角', '#C44E52'),
        (3, 1.0, '→ 全原子坐标 + pLDDT', '#8172B2'),
    ]
    for x, y, label, color in blocks2:
        ax.add_patch(mpatches.FancyBboxPatch((x-2.2, y-0.5), 4.4, 0.9,
                     boxstyle='round,pad=0.05', facecolor=color, alpha=0.8))
        ax.text(x, y, label, ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    ax.set_title('Structure Module（结构模块）', fontsize=11)
    ax.axis('off')

    plt.suptitle('AlphaFold 2 核心架构（2021）', fontsize=13)
    plt.tight_layout()
    plt.show()


plot_af2_pipeline()

## 5. MSA（多序列比对）的作用

In [ ]:
# 用模拟 MSA 可视化共进化信号
np.random.seed(42)

seq_len = 30
n_seqs  = 20

# 模拟 MSA：相同进化压力导致某些位置协同变化
msa = np.random.randint(0, 20, (n_seqs, seq_len)).astype(float)
# 引入共进化信号：位置 5 和 15 协同突变
for i in range(n_seqs):
    if msa[i, 5] > 10:
        msa[i, 15] = msa[i, 5] * 0.8 + np.random.randn() * 2

# 计算列间相关（粗略 MI 代理）
corr = np.corrcoef(msa.T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(msa, aspect='auto', cmap='tab20')
axes[0].set_xlabel('序列位置')
axes[0].set_ylabel('同源序列')
axes[0].set_title('模拟 MSA（多序列比对）')

im1 = axes[1].imshow(np.abs(corr), cmap='hot', vmin=0, vmax=1)
axes[1].set_xlabel('残基位置')
axes[1].set_ylabel('残基位置')
axes[1].set_title('列间相关矩阵（共进化信号）')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()
print('共进化高相关位置对提示这两个残基在三维空间中可能接近（形成接触）')

## 6. 使用 ESMFold 做本地推理

ESMFold（Meta AI）是单序列蛋白质折叠模型，无需 MSA 检索，可在本地 GPU 上运行，结构精度接近 AlphaFold 2。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
esm_model = EsmForProteinFolding.from_pretrained(cfg.model_name, low_cpu_mem_usage=True)
esm_model = esm_model.to(device)
esm_model.eval()

# 将主干切换到 bf16 节省显存（Ampere 以上 GPU 支持）
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    esm_model = esm_model.bfloat16()

print('ESMFold 加载完成')

In [ ]:
@torch.no_grad()
def predict_structure(model, tokenizer, sequence, device):
    inputs = tokenizer(
        [sequence],
        return_tensors='pt',
        add_special_tokens=False,
    ).to(device)

    outputs = model(**inputs)
    # pLDDT：每个残基的预测局部距离差异测试，越高越可信（100 为满分）
    plddt = outputs.plddt[0].cpu().float().numpy()  # (L,)
    # 预测的 Cα 原子坐标
    positions = outputs.positions[-1, 0].cpu().float().numpy()  # (L, 37, 3) 全原子
    ca_coords  = positions[:, 1, :]  # Cα 是原子索引 1

    return plddt, ca_coords


print('正在预测 Trp-cage 结构（20 aa，约需 10-30 秒）...')
plddt_trp, ca_trp = predict_structure(esm_model, tokenizer, TRP_CAGE, device)
print(f'预测完成！平均 pLDDT = {plddt_trp.mean():.1f}')

print('\n正在预测 Ubiquitin 结构（76 aa，约需 30-60 秒）...')
plddt_ubi, ca_ubi = predict_structure(esm_model, tokenizer, UBIQUITIN, device)
print(f'预测完成！平均 pLDDT = {plddt_ubi.mean():.1f}')

## 7. 结构可视化（pLDDT 着色）

pLDDT（predicted Local Distance Difference Test）是 AlphaFold 系列对每个残基预测置信度的指标：

| pLDDT 范围 | 颜色 | 含义 |
|-----------|------|------|
| ≥ 90 | 蓝色 | 非常高置信度 |
| 70–90 | 青色 | 置信度良好 |
| 50–70 | 黄色 | 低置信度（可能无序区域） |
| < 50 | 橙色 | 非常低（可能是 IDR）|

In [ ]:
def plot_plddt(plddt, sequence, title):
    fig, ax = plt.subplots(figsize=(max(8, len(sequence) * 0.25), 3))
    colors = []
    for p in plddt:
        if p >= 90:   colors.append('#0053D6')
        elif p >= 70: colors.append('#65CBF3')
        elif p >= 50: colors.append('#FFDB13')
        else:         colors.append('#FF7D45')

    bars = ax.bar(range(len(plddt)), plddt, color=colors, width=0.8)
    ax.axhline(90, color='#0053D6', linestyle='--', alpha=0.5, label='≥90 非常高')
    ax.axhline(70, color='#65CBF3', linestyle='--', alpha=0.5, label='≥70 良好')
    ax.axhline(50, color='#FFDB13', linestyle='--', alpha=0.5, label='≥50 低')
    ax.set_xticks(range(len(sequence)))
    ax.set_xticklabels(list(sequence), fontsize=7)
    ax.set_ylabel('pLDDT 置信度')
    ax.set_ylim(0, 100)
    ax.set_title(f'{title}  (均值={plddt.mean():.1f})')
    ax.legend(loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()


plot_plddt(plddt_trp, TRP_CAGE,  'Trp-cage pLDDT')
plot_plddt(plddt_ubi, UBIQUITIN, 'Ubiquitin pLDDT')

In [ ]:
def plot_3d_backbone(ca_coords, plddt, title):
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    # 用 pLDDT 着色 Cα 轨迹
    norm = plt.Normalize(plddt.min(), plddt.max())
    cmap = plt.cm.coolwarm

    for i in range(len(ca_coords) - 1):
        xs = ca_coords[i:i+2, 0]
        ys = ca_coords[i:i+2, 1]
        zs = ca_coords[i:i+2, 2]
        color = cmap(norm((plddt[i] + plddt[i+1]) / 2))
        ax.plot(xs, ys, zs, color=color, linewidth=2)

    sc = ax.scatter(ca_coords[:, 0], ca_coords[:, 1], ca_coords[:, 2],
                    c=plddt, cmap='coolwarm', s=30, zorder=5)
    plt.colorbar(sc, ax=ax, label='pLDDT', shrink=0.6)
    ax.set_title(f'{title}（Cα 骨架，pLDDT 着色）')
    ax.set_xlabel('X (Å)')
    ax.set_ylabel('Y (Å)')
    ax.set_zlabel('Z (Å)')
    plt.tight_layout()
    plt.show()


plot_3d_backbone(ca_trp, plddt_trp, 'Trp-cage')
plot_3d_backbone(ca_ubi, plddt_ubi, 'Ubiquitin')

## 8. 接触图预测与分析

In [ ]:
def compute_contact_map(ca_coords, threshold=8.0):
    # 以 Cα 间距 ≤ 8 Å 定义接触
    L = len(ca_coords)
    dist = np.zeros((L, L))
    for i in range(L):
        for j in range(L):
            dist[i, j] = np.linalg.norm(ca_coords[i] - ca_coords[j])
    contact = dist <= threshold
    return dist, contact


dist_ubi, contact_ubi = compute_contact_map(ca_ubi)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(dist_ubi, cmap='hot_r')
axes[0].set_title('Ubiquitin Cα 距离矩阵 (Å)')
axes[0].set_xlabel('残基编号')
axes[0].set_ylabel('残基编号')
plt.colorbar(im0, ax=axes[0], label='距离 (Å)')

axes[1].imshow(contact_ubi, cmap='Blues')
axes[1].set_title('Ubiquitin 接触图（Cα < 8 Å）')
axes[1].set_xlabel('残基编号')
axes[1].set_ylabel('残基编号')

plt.tight_layout()
plt.show()
print('对角线附近的接触对应局部二级结构（螺旋/折叠），远离对角线的接触对应长程折叠相互作用')

## 9. AlphaFold 的范式意义

### 9.1 对生物学的冲击

- AlphaFold 2 发布后，DeepMind 预测了人类蛋白质组全部约 20,000 个蛋白的结构，并公开在 **AlphaFold DB**（alphafold.ebi.ac.uk）
- 截至 2024 年，数据库已覆盖超过 **2 亿个**蛋白质结构
- 极大加速了药物设计、酶工程、疫苗开发等领域

### 9.2 后续工作

| 模型 | 发布 | 进步 |
|------|------|------|
| AlphaFold 2 | 2021 | 氨基酸链折叠，CASP14 冠军 |
| ESMFold | 2022 | 单序列折叠，无需 MSA |
| RoseTTAFold | 2021 | 三轨道网络，支持多链 |
| AlphaFold 3 | 2024 | 扩展到蛋白质-DNA/RNA/小分子复合物 |

### 9.3 核心 Takeaway

AlphaFold 1 → AlphaFold 2 的本质演进：
**CNN 特征提取 → Transformer 全局建模**，与视觉领域 CNN → ViT 的演进方向完全一致。
深度学习的成功范式在生物学中同样成立：数据规模 + 归纳偏置 + 端到端训练 = 质变。